# H4 Composition

Frozen before the run.

**Claim.** Orchestrator reasoning makes up for separate encoding at 1 crossing, but not at 2.

**Arms** (300 per cell, seed 0): B1 and K0, each with thinking off, and with thinking forced (`<think>` prefilled in every arm, 1024-token budget, then `</think>` is forced). Forcing matters: asking for reasoning gets ignored in long prompts, which would make reasoning depend on the channel.

**Prediction.** Recovery = (B1 − K0 | off) − (B1 − K0 | forced):
- Positive at 1 crossing.
- ≈ 0 at 2+ crossings.

Thinking length should be similar across B1 and K0 at the same budget. If it isn't, the comparison is confounded.

**Diagnostic.** Does K0's thinking trace name the gold bridge entity? This separates two failures:
- the orchestrator never linked the blocks;
- it linked them but failed the last step.

**Pilot (Qwen2.5-7B toy, n = 100, seed 0, EM):**

| | none | asked | forced |
|---|---|---|---|
| single, 1 crossing | 96 | 99 | 99 |
| merge, 1 crossing | 32 | 68 | 85 |
| single, 2 crossings | 56 | 80 | 97 |
| merge, 2 crossings | 30 | 32 | 32 |

MuSiQue with forced reasoning: merge was still −.12 F1.

In [ ]:
# Settings. Headless full run: see README "Running".
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SMOKE = True        # True: 2 questions per cell, to read raw outputs first. False: the full design.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from latent_dispatch import analysis, arms, data
from latent_dispatch.model import Model
from latent_dispatch.results import load_results, results_path, run
from latent_dispatch.score import contains_answer, normalize

## Questions and arms

Forced thinking costs up to 1024 generated tokens per question, so 300 per cell, seed 0. `b1` and `k0` (thinking off) are shared with H1.

In [ ]:
qs = data.load("toy", seed=0) + data.load("toy_d0", seed=0) + data.load("musique", seed=0)
qs = data.per_cell(qs, 2 if SMOKE else 300)
print(len(qs), "questions")

FORCED = {"thinking": "forced", "think_budget": 1024}
ARMS = {
    "b1": (arms.b1, {}),
    "k0": (arms.k0, {}),
    "b1_forced": (arms.b1, FORCED),
    "k0_forced": (arms.k0, FORCED),
}

In [ ]:
model = Model.load(MODEL, dtype=DTYPE)
for name, (arm, options) in ARMS.items():
    run(model, qs, name, arm, **options)

In [ ]:
df = load_results(qs, ARMS, MODEL, DTYPE)
if SMOKE:
    analysis.peek(df)

## Accuracy

In [ ]:
analysis.accuracy(df)

## Recovery = (B1 − K0 | off) − (B1 − K0 | forced), by crossings

In [ ]:
from latent_dispatch import stats

w = analysis.wide(df, "em", list(ARMS)).dropna(subset=list(ARMS))
w["gap_off"] = w.b1 - w.k0
w["gap_forced"] = w.b1_forced - w.k0_forced
rows = []
for (ds, c), g in w.groupby(["dataset", "crossings"]):
    est, lo, hi = stats.bootstrap_ci((g.gap_off - g.gap_forced).to_numpy())
    rows.append({"dataset": ds, "crossings": c, "n": len(g), "gap_off": 100 * g.gap_off.mean(),
                 "gap_forced": 100 * g.gap_forced.mean(), "recovery": 100 * est, "lo": 100 * lo, "hi": 100 * hi})
recovery = pd.DataFrame(rows).set_index(["dataset", "crossings"]).round(1)
recovery

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
for ds, g in recovery.reset_index().groupby("dataset"):
    ax.errorbar(g.crossings, g.recovery, yerr=[g.recovery - g.lo, g.hi - g.recovery], marker="o", capsize=3, label=ds)
ax.axhline(0, color="gray", lw=0.8)
ax.set(xlabel="crossings", ylabel="recovery (EM points, 95% CI)")
ax.legend()
plt.show()

## Thinking length, and is the bridge named in the trace? (split questions answered wrong)

In [ ]:
rows = []
for arm in ("b1_forced", "k0_forced"):
    a = df[df.arm == arm]
    wrong = a[(a.crossings > 0) & (a.em == 0)]
    named = [contains_answer(t.split("</think>")[0], [b]) for t, bs in zip(wrong.text, wrong.bridges) for b in bs]
    rows.append({"arm": arm, "mean think tokens": round(a.think_tokens.mean()) if len(a) else None,
                 "bridge named (split, wrong)": f"{100 * sum(named) / len(named):.1f}%" if named else "-"})
pd.DataFrame(rows)

## Findings

_(fill in after the full run)_